# 1. Draft the training answers with a bigger "teacher" model

Runtime: **Runtime > Change runtime type > T4 GPU** (free).

This makes about 1,200 fake folder tasks and lets a bigger Qwen model (the teacher) write the ideal answer for each.
Answers that break a safety rule are rejected automatically. You then review the drafts on your PC with
`python dataset/review.py` before anything is used for training.

The full run takes several hours. If Colab disconnects, run the cells again: it continues where it stopped.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Unzip local-assistant.zip into your Google Drive once, so it is at MyDrive/local-assistant.
# Everything this notebook writes goes there too, so nothing is lost if Colab disconnects.
PROJECT = "/content/drive/MyDrive/local-assistant"
%cd {PROJECT}
!ls

In [ ]:
# Install Ollama and the teacher model
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)

TEACHER = "qwen3.5:9b"   # fits the free T4. On a 24 GB+ GPU (L4, A100) "qwen3.5:27b" gives better answers.
!ollama pull {TEACHER}
!pip install -q ollama pypdf python-docx pillow send2trash

In [ ]:
# Make the tasks once (fake folders + requests). Delete dataset/tasks.jsonl to make new ones.
import os
if not os.path.exists("dataset/tasks.jsonl"):
    !python dataset/make_scenarios.py --scenarios 300 --requests 4
!wc -l dataset/tasks.jsonl

In [ ]:
# Quick try: 10 drafts. Look at one before starting the long run.
!python dataset/draft_answers.py --teacher {TEACHER} --limit 10
!python dataset/validate.py dataset/drafts.jsonl
import json
first = json.loads(open("dataset/drafts.jsonl", encoding="utf-8").readline())
for m in first["messages"][1:]:
    print(m["role"].upper(), ":", (m.get("content") or "")[:400], m.get("tool_calls", ""), "\n")

In [ ]:
# The full run. Re-run this cell after a disconnect; finished tasks are skipped.
!python dataset/draft_answers.py --teacher {TEACHER}

In [ ]:
# What was rejected and why
import json
from collections import Counter
ok = sum(1 for _ in open("dataset/drafts.jsonl", encoding="utf-8"))
bad = [json.loads(l) for l in open("dataset/rejected.jsonl", encoding="utf-8")] if os.path.exists("dataset/rejected.jsonl") else []
print(f"{ok} drafts to review, {len(bad)} rejected")
print(Counter(e.split(":")[-1].strip() for r in bad for e in r.get("errors", [])).most_common(10))

## Next, on your PC
1. Copy `dataset/drafts.jsonl` from Google Drive into `local-assistant\dataset\` on your PC.
2. `python dataset/review.py`: accept, edit or reject each draft (Enes and Kaan can split the work).
3. `python dataset/split.py`: makes `train.jsonl`, `val.jsonl`, `test.jsonl`.
4. Copy those three files back to `MyDrive/local-assistant/dataset/` and open `2_finetune.ipynb`.